In [1]:

from pathlib import Path
import json
OUT=Path(r'C:/Users/choud/Desktop/Minor project/model_improvement/attention_tuning_v1');n=json.loads((OUT/'01_tuning.ipynb').read_text());exec(compile(''.join(n['cells'][0]['source']),'tuning_definitions_only','exec'))
study=torch.load(OUT/'models/selected_3s_study.pt',map_location='cpu',weights_only=False);ck=study['scalers'];original_ck=torch.load(ORIGINAL/'models/attention_3s_best.pt',map_location='cpu',weights_only=False)
assert all(sha(Path(p))==v for p,v in study['input_hashes'].items())
# Final choice is immutable before opening test inputs. No untouched set is available in the authorized verified artifacts.
with h5py.File(V3/'dataset/eew_dataset_test.h5','r') as w,h5py.File(TAB,'r') as t:
 wp=w['parent_id'].asstr()[:];tp=t['test/parent_id'].asstr()[:];lookup={p:i for i,p in enumerate(tp)};assert set(wp)==set(tp) and len(set(wp))==len(wp);MAP['test']=np.array([lookup[p] for p in wp])
 for key in ['source_id','labels','magnitude','classification_mask_3s','magnitude_mask_3s']:
  aa=w[key].asstr()[:] if w[key].dtype.kind=='O' else w[key][:];bb=t['test'][key].asstr()[:] if t['test'][key].dtype.kind=='O' else t['test'][key][:];assert np.array_equal(aa,bb[MAP['test']])
c,b,cc,bc=branches(3);d=temporal('test',3,c,b,True);base=encode('test',3,c,b,cc,bc,True);assert np.array_equal(d['ids'],base['ids']);assert int(d['reg'].sum())==1200 and len(d['y'])==2414
assert hashlib.sha256('\n'.join(d['ids']).encode()).hexdigest()=='862b467ff6219c5193f4c99257a5971eea395581669302748d006faa78b78601'
selected=TunedHead(study['selected_config']['hidden'],study['selected_config']['dropout']).to(DEV) if study['selected_trial'] is not None else AttentionHead().to(DEV);selected.load_state_dict(study['selected_state']);original=AttentionHead().to(DEV);original.load_state_dict(original_ck['head_state'])
p,m,_=predict_attention(selected,d,ck);op,om,_=predict_attention(original,d,original_ck);y=d['y'].numpy();actual=d['m'].numpy();reg=d['reg'].numpy()
# Thresholds frozen on validation; no test threshold optimization.
threshold=study['selected_validation']['operating']['threshold'];ot=study['baseline_validation']['operating']['threshold'];results={}
for name,prob,pred,t in [('Selected',p,m,threshold),('Original Attention',op,om,ot),('CNN',base['cnn_p'],base['cnn_m'],None)]:
 _,mm=stats(d,prob,pred);results[name]={'fixed_05':det(y,prob,.5),'validation_operating_test':det(y,prob,t) if t is not None else None,'validation_threshold':t,'magnitude':mm}
assert abs(results['Original Attention']['fixed_05']['f1']-.7474275023386342)<1e-6
# Validation CNN comparison uses same eligible validation cohort, not test-driven thresholds.
va=encode('validation',3,c,b,cc,bc);cnnval={'fixed_05':det(va['y'].numpy(),va['cnn_p'],.5),'magnitude':magnitude(va['m'].numpy()[va['reg'].numpy()],va['cnn_m'][va['reg'].numpy()])}
def paired(bp,bm,threshold_a=.5,threshold_b=.5):
 rng=np.random.default_rng(2026);eq=np.flatnonzero(reg);noise=np.flatnonzero(~reg);df=[];dm=[]
 def f1(y,p,t):
  yes=y==1;q=p>=t;tp=(yes&q).sum();return float(2*tp/(2*tp+(~yes&q).sum()+(yes&~q).sum()))
 for _ in range(500):
  e=rng.choice(eq,len(eq),replace=True);n=rng.choice(noise,len(noise),replace=True);ix=np.r_[e,n];df.append(f1(y[ix],p[ix],threshold_a)-f1(y[ix],bp[ix],threshold_b));dm.append(float((abs(m[e]-actual[e])-abs(bm[e]-actual[e])).mean()))
 return {'f1_difference_ci95':np.quantile(df,[.025,.975]).tolist(),'mae_difference_ci95':np.quantile(dm,[.025,.975]).tolist()}
comparisons={'original_fixed':paired(op,om),'cnn_fixed':paired(base['cnn_p'],base['cnn_m']),'original_validation_operating':paired(op,om,threshold,ot)}
print('Final selection',study['selected_trial'],'test results',json.dumps(results),flush=True)


Exact paired inputs; frozen temporal backbone states available.
Final selection None test results {"Selected": {"fixed_05": {"accuracy": 0.776304888152444, "precision": 0.8518123667377399, "recall": 0.6658333333333334, "f1": 0.7474275023386342, "false_positive_rate": 0.11449752883031301, "roc_auc": 0.865392641405821, "pr_auc_ap": 0.887531712083913, "confusion_matrix": [[1075, 139], [401, 799]]}, "validation_operating_test": {"accuracy": 0.7618061309030655, "precision": 0.7715030408340573, "recall": 0.74, "f1": 0.7554232241599319, "false_positive_rate": 0.21663920922570015, "roc_auc": 0.865392641405821, "pr_auc_ap": 0.887531712083913, "confusion_matrix": [[951, 263], [312, 888]]}, "validation_threshold": 0.41, "magnitude": {"n": 1200, "mae": 0.6066006402174632, "rmse": 0.7486207347412538, "r2": 0.30672507114512293, "bias": -0.004880607922871908, "actual_variance": 0.8083849294972121, "predicted_variance": 0.29162310351765447, "std_ratio": 0.6006228625923995, "slope": 0.3337511803935037,

In [2]:

plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
def save(fig,name):
 path=OUT/'plots'/name;assert not path.exists();fig.tight_layout();fig.savefig(path,dpi=300);plt.close(fig)
fig,axs=plt.subplots(1,2,figsize=(11,4));curve=study['baseline_validation']['curve']
for key in ['precision','recall','f1','false_positive_rate']:axs[0].plot([q['threshold'] for q in curve],[q[key] for q in curve],label=key)
axs[0].axhline(.30,ls=':',color='black');axs[0].set(xlabel='Threshold',ylabel='Validation metric',title='Original3s Attention threshold analysis');axs[0].legend(fontsize=8)
for label,v in [('Original',study['baseline_validation']),('Selected',study['selected_validation'])]:
 q=v['curve'];axs[1].plot([x['false_positive_rate'] for x in q],[x['recall'] for x in q],label=label)
axs[1].set(xlabel='Validation FPR',ylabel='Validation recall',title='Validation operating tradeoff');axs[1].legend();save(fig,'validation_threshold_tradeoffs.png')
fig,axs=plt.subplots(1,2,figsize=(11,4))
for r in study['trials']:
 q=r['operating'];mm=r['magnitude'];axs[0].scatter(q['false_positive_rate'],q['f1']);axs[0].annotate(str(r['config']['trial']),(q['false_positive_rate'],q['f1']));axs[1].scatter(mm['mae'],mm['strong_mae']);axs[1].annotate(str(r['config']['trial']),(mm['mae'],mm['strong_mae']))
axs[0].set(xlabel='Validation FPR',ylabel='Validation F1',title='Predeclared operating points');axs[1].set(xlabel='Overall validation MAE',ylabel='M>=5 validation MAE',title='Magnitude tradeoffs (trial labels)');save(fig,'validation_trial_tradeoffs.png')
fig,axs=plt.subplots(1,2,figsize=(10,4))
for ax,pred,label in zip(axs,[om,m],['Original Attention','Selected']):
 ax.scatter(actual[reg],pred[reg],s=8,alpha=.3);ax.plot([actual[reg].min(),actual[reg].max()],[actual[reg].min(),actual[reg].max()],'k--');ax.set(xlabel='True magnitude',ylabel='Predicted magnitude',title=label+' exploratory test')
save(fig,'exploratory_magnitude_comparison.png')
lines=['# Controlled3s Attention tuning study','','## Scope and preregistered protocol','','Only3s studied. Existing backbones frozen and unchanged. All trial fitting and selection used training/validation only; no test prediction used until final checkpoint selection. Existing reports/checkpoints contain no stored validation probability archive, so Phase A probabilities were reproduced from the frozen original checkpoint. Existing synthetic oracle-aligned data reused without regeneration.','', 'Protocol recorded in the executed tuning notebook before trial execution:','', '```json',json.dumps(study['protocol'],indent=2),'```','', 'Ten fixed Phase-B configurations cover learning rates1e-4/3e-4/1e-3, dropout0.1–0.4, hidden64/128, regression weights0.5/1/2. Up to two Phase-C variants are included in the12-trial cap, not extra trials. Seed42, batch256, AdamW weight decay1e-4, clipping5, maximum35 epochs/patience7. Original train-fitted hidden-state/CNN and target scalers exactly match moments recomputed from training records. Selection is gated lexicographic, not a hidden aggregate score. Epoch checkpoint selection uses the declared unweighted validation BCE+Huber criterion.','', '## Phase A: original validation operating points','','Validation cohort:1,203 eligible earthquakes +1,213 noise; distinct from test. FPR<=30% alone can be achieved by rejecting all examples, so useful recall>=70% is explicitly required.','', '| Policy | Threshold | Precision | Recall | F1 | FPR |','|---|---:|---:|---:|---:|---:|']
bl=study['baseline_validation']
for label,q,t in [('Fixed',bl['fixed'],.5),('Unconstrained validation F1',bl['unconstrained'],bl['unconstrained']['threshold']),('Constrained validation operating point',bl['operating'],bl['operating']['threshold'])]:lines.append(f"| {label} | {t:.4f} | {q['precision']:.6f} | {q['recall']:.6f} | {q['f1']:.6f} | {q['false_positive_rate']:.6f} |")
lines+=['',f"Useful target feasible on validation: {bl['feasible']}. A validation constraint is not a test or deployment FPR guarantee.",'', '## Phase B/C trials and final selection','',f"Trials executed: {len(study['trials'])}. Phase-B reference for loss variants: trial{study['phase_b_reference_trial']}. Selected trial: {study['selected_trial']}; None means original retained. Configuration: `{json.dumps(study['selected_config'])}`.",'', '| Trial | LR | Dropout | Hidden | Reg weight | Variant | Best epoch | F1 | FPR | Recall | MAE | Strong MAE | Underprediction % |','|---|---:|---:|---:|---:|---|---:|---:|---:|---:|---:|---:|---:|']
for r in study['trials']:
 cfg=r['config'];q=r['operating'];mm=r['magnitude'];lines.append(f"| {cfg['trial']} | {cfg['lr']} | {cfg['dropout']} | {cfg['hidden']} | {cfg['reg_weight']} | {cfg['variant']} | {r['best_epoch']} | {q['f1']:.6f} | {q['false_positive_rate']:.6f} | {q['recall']:.6f} | {mm['mae']:.6f} | {mm['strong_mae']:.6f} | {mm['strong_underprediction_percent']:.3f} |")
lines+=['','Full trial MAE/RMSE/R2/bias/compression and fixed-.5 detection values are in trial_results.csv. Trial histories and complete validation threshold curves are retained in the single necessary selected study checkpoint; discarded trial weights were not saved. Magnitude variants use true magnitude only to weight training losses, never as an input feature.','', '## Validation magnitude comparison','','| Model | MAE | RMSE | R2 | Bias | SD ratio | Strong MAE | Underprediction % |','|---|---:|---:|---:|---:|---:|---:|---:|']
for name,v in [('Original Attention',bl),('Selected',study['selected_validation'])]:
 mm=v['magnitude'];lines.append(f'| {name} | '+' | '.join(f'{mm[k]:.6f}' for k in ['mae','rmse','r2','bias','std_ratio','strong_mae','strong_underprediction_percent'])+' |')
lines+=['',f"CNN validation fixed-.5 metrics and magnitude (same eligible validation cohort): `{json.dumps(cnnval)}`.",'', '## One final exploratory test comparison','','No authorized untouched evaluation set was supplied; no independent data collection or project scan was performed. The existing test cohort has been used in previous studies. This is therefore exploratory, not an unbiased confirmation. Exactly1,200 earthquakes +1,214 noise with the audited ordered identity hash. Both selected/original predictions computed only after selection; CNN frozen predictions provide the requested reference.','', '| Model | Policy | Threshold | Precision | Recall | F1 | FPR | MAE | RMSE | R2 | M>=5 underprediction % |','|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
for name,v in results.items():
 for policy,q,t in [('Fixed',v['fixed_05'],.5)]+([] if v['validation_operating_test'] is None else [('Validation operating point',v['validation_operating_test'],v['validation_threshold'])]):
  mm=v['magnitude'];lines.append(f"| {name} | {policy} | {t:.4f} | {q['precision']:.6f} | {q['recall']:.6f} | {q['f1']:.6f} | {q['false_positive_rate']:.6f} | {mm['mae']:.6f} | {mm['rmse']:.6f} | {mm['r2']:.6f} | {mm['strong_underprediction_percent']:.3f} |")
lines+=['','Same-cohort magnitude covers every eligible earthquake independent of threshold; noise excluded. Fixed-.5 CNN comparisons are separate from validation-selected operating points; no CNN threshold was tuned.','', 'Paired500-resample stratified bootstrap, seed2026, selected minus baseline; same event draws per model; descriptive percentile95% intervals (not corrected for search or historical test reuse):','', '```json',json.dumps(comparisons,indent=2),'```','', '## Interpretation','', ('No trial met the predeclared useful-improvement gates; original Attention retained. Tuning did not establish useful validation improvement.' if study['selected_trial'] is None else 'A candidate met the predeclared validation improvement gates. This is a validation-selected tradeoff, not proof of general superiority. Evaluate fixed-.5 and operating-point results separately; any test deterioration is retained without reselection.'),'', 'Generalizable improvement remains uncertain: single seed, small validation cohort, up to12 comparisons, known synthetic/test limitations. Strong-event loss weighting can reduce underprediction while increasing overall bias/error; guards prevent selection solely on strong-event scores. Check numerical tables rather than assuming that larger heads or weighted losses improve all objectives.','', '## Limitations and preservation','','Synthetic oracle-P-aligned benchmark; both branches derive from the same simulated waveform source. No real-world triggering, warning lead time or operational EEW readiness established. High false positives, magnitude compression and strong-event underprediction remain relevant. Bidirectional temporal states use only the observed400-sample/40-bin horizon. Validation-derived thresholds do not guarantee deployment false-alarm rates.','', 'Only 01_tuning.ipynb, 02_tuning_evaluation.ipynb, trial_results.csv, models/, plots/ and tuning_report.md created. No1s/2s training, cleanup or changes to approved experiments. Checkpoint includes source SHA256 hashes.','']
changed=[p for p,v in study['input_hashes'].items() if sha(Path(p))!=v];assert not changed;lines.append(f"All {len(study['input_hashes'])} original dependencies unchanged; pretrained tensors remained exactly equal after training. Changed files: {changed}.")
lines+=['','Complete final test values:','','```json',json.dumps(results,indent=2),'```']
with (OUT/'tuning_report.md').open('x',encoding='utf-8') as f:f.write('\n'.join(lines)+'\n')
print('Evaluation and report complete; originals unchanged.')


Evaluation and report complete; originals unchanged.
